# 类别编码(categorical_encoding)
***

In [16]:
import pandas as pd
import numpy as np

In [17]:
vg_df = pd.read_csv("../datasets/vgsales.csv",encoding= "ISO-8859-1")
vg_df[['Name','Platform','Year','Genre','Publisher']].iloc[1:7]

,Name,Platform,Year,Genre,Publisher
1,Super Mario Bros.,NES,1985.0,Platform,Nintendo
2,Mario Kart Wii,Wii,2008.0,Racing,Nintendo
3,Wii Sports Resort,Wii,2009.0,Sports,Nintendo
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,Nintendo
5,Tetris,GB,1989.0,Puzzle,Nintendo
6,New Super Mario Bros.,DS,2006.0,Platform,Nintendo


### 1. LabelEncoder

* **使用Sklearn的LabelEncoder**

In [18]:
genres = np.unique(vg_df['Genre'])
genres

array(['Action', 'Adventure', 'Fighting', 'Misc', 'Platform', 'Puzzle',
       'Racing', 'Role-Playing', 'Shooter', 'Simulation', 'Sports',
       'Strategy'], dtype=object)

In [19]:
from sklearn.preprocessing import LabelEncoder

gle = LabelEncoder()
genre_labels = gle.fit_transform(vg_df["Genre"])   #只吃一维数据
genre_mappings = {index:label for index, label in enumerate(gle.classes_)}
genre_mappings

{0: 'Action',
 1: 'Adventure',
 2: 'Fighting',
 3: 'Misc',
 4: 'Platform',
 5: 'Puzzle',
 6: 'Racing',
 7: 'Role-Playing',
 8: 'Shooter',
 9: 'Simulation',
 10: 'Sports',
 11: 'Strategy'}

In [20]:
vg_df['GenreLabel'] = genre_labels
vg_df[['Name','Platform','Year','Genre','GenreLabel']].iloc[1:7]

,Name,Platform,Year,Genre,GenreLabel
1,Super Mario Bros.,NES,1985.0,Platform,4
2,Mario Kart Wii,Wii,2008.0,Racing,6
3,Wii Sports Resort,Wii,2009.0,Sports,10
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,7
5,Tetris,GB,1989.0,Puzzle,5
6,New Super Mario Bros.,DS,2006.0,Platform,4


### 2. Series.map(dict)

In [24]:
platform = np.unique(vg_df['Platform'])
platform

array(['2600', '3DO', '3DS', 'DC', 'DS', 'GB', 'GBA', 'GC', 'GEN', 'GG',
       'N64', 'NES', 'NG', 'PC', 'PCFX', 'PS', 'PS2', 'PS3', 'PS4', 'PSP',
       'PSV', 'SAT', 'SCD', 'SNES', 'TG16', 'WS', 'Wii', 'WiiU', 'X360',
       'XB', 'XOne'], dtype=object)

In [27]:
pltf_old_map ={'2600':1, '3DO':2, '3DS':3, 'DC':4, 'DS':5, 'GB':6, 'GBA':7, 'GC':8, 'GEN':9, 'GG':10,
       'N64':11, 'NES':12, 'NG':13, 'PC':14, 'PCFX':15, 'PS':16, 'PS2':17, 'PS3':18, 'PS4':19, 'PSP':20,
       'PSV':21, 'SAT':22, 'SCD':23, 'SNES':24, 'TG16':25, 'WS':26, 'Wii':27, 'WiiU':28, 'X360':29,
       'XB':30, 'XOne':31}

vg_df['PlatformLabel'] = vg_df['Platform'].map(pltf_old_map)
vg_df[['Name','Platform','PlatformLabel','Year','Genre']].iloc[1:7]

,Name,Platform,PlatformLabel,Year,Genre
1,Super Mario Bros.,NES,12,1985.0,Platform
2,Mario Kart Wii,Wii,27,2008.0,Racing
3,Wii Sports Resort,Wii,27,2009.0,Sports
4,Pokemon Red/Pokemon Blue,GB,6,1996.0,Role-Playing
5,Tetris,GB,6,1989.0,Puzzle
6,New Super Mario Bros.,DS,5,2006.0,Platform


### 3. One-hot Encoding

In [31]:
from sklearn.preprocessing import OneHotEncoder

genre_encoder = OneHotEncoder(sparse_output=False)
genre_arr = genre_encoder.fit_transform(vg_df[["Genre"]])    # 双括号：二维输入
genre_labels = list(genre_encoder.categories_[0])        
genre = pd.DataFrame(genre_arr,
                     columns='Genre_' + pd.Index(genre_labels),   # 加前缀防重名
                     index=vg_df.index)                       

vg_df_one = pd.concat([vg_df, genre], axis=1)

columns = ['Name', 'Platform', 'Year', 'Genre'] + list(genre.columns) 
vg_df_one[columns].iloc[1:7]                                     

,Name,Platform,Year,Genre,Genre_Action,Genre_Adventure,Genre_Fighting,Genre_Misc,Genre_Platform,Genre_Puzzle,Genre_Racing,Genre_Role-Playing,Genre_Shooter,Genre_Simulation,Genre_Sports,Genre_Strategy
1,Super Mario Bros.,NES,1985.0,Platform,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Mario Kart Wii,Wii,2008.0,Racing,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3,Wii Sports Resort,Wii,2009.0,Sports,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,Pokemon Red/Pokemon Blue,GB,1996.0,Role-Playing,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
5,Tetris,GB,1989.0,Puzzle,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
6,New Super Mario Bros.,DS,2006.0,Platform,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### 4. Get Dummy

In [34]:
genre_dummy = pd.get_dummies(vg_df['Genre'],drop_first=True)   # 把第一列Action丢了，因为全零就是它，在线性模型（逻辑回归等）需要，避免共线
pd.concat([vg_df[['Name','Genre']],genre_dummy],axis=1).iloc[1:7]

,Name,Genre,Adventure,Fighting,Misc,Platform,Puzzle,Racing,Role-Playing,Shooter,Simulation,Sports,Strategy
1,Super Mario Bros.,Platform,False,False,False,True,False,False,False,False,False,False,False
2,Mario Kart Wii,Racing,False,False,False,False,False,True,False,False,False,False,False
3,Wii Sports Resort,Sports,False,False,False,False,False,False,False,False,False,True,False
4,Pokemon Red/Pokemon Blue,Role-Playing,False,False,False,False,False,False,True,False,False,False,False
5,Tetris,Puzzle,False,False,False,False,True,False,False,False,False,False,False
6,New Super Mario Bros.,Platform,False,False,False,True,False,False,False,False,False,False,False
